In [117]:
import os
from langchain_core.tools import tool, InjectedToolArg
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
import requests
from pprint import pprint

In [118]:
key = os.environ.get("OPENAI_API_KEY")

In [119]:
llm = ChatOpenAI()

In [ ]:
# tool creation

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency.
    """
    url = f"https://v6.exchangerate-api.com/v6/499893dd52e0fccc5d912c39/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    data = response.json()
    return data["conversion_rate"]


In [121]:
conversion_rate = get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})
print(conversion_rate)

95.7625


In [122]:
# tool for multiplication

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value.
    """
    return base_currency_value * conversion_rate

In [123]:
convert.invoke({"base_currency_value": 100, "conversion_rate": conversion_rate})

9576.25

In [124]:
# tool binding

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [125]:
messages = [
    HumanMessage("What is the conversion factor between USD and INR? And based on that can you convert 10 usd to inr?")
]

print(messages)

[HumanMessage(content='What is the conversion factor between USD and INR? And based on that can you convert 10 usd to inr?', additional_kwargs={}, response_metadata={})]


In [126]:
ai_message = llm_with_tools.invoke(messages)

In [127]:
messages.append(ai_message)

In [128]:
pprint(ai_message.tool_calls)

[{'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_3wSAm8EGMlZQ3k7rp1ObslhU',
  'name': 'get_conversion_factor',
  'type': 'tool_call'},
 {'args': {'base_currency_value': 10},
  'id': 'call_zqZ9MZpUo0HK22GRbHkOgSg2',
  'name': 'convert',
  'type': 'tool_call'}]


In [129]:
tool_registry = {
    "get_conversion_factor": get_conversion_factor,
    "convert": convert,
}

for tool_call in ai_message.tool_calls:
    tool_name = tool_call["name"]
    tool_args = dict(tool_call["args"])

    # Inject the conversion rate for the convert tool when needed.
    if tool_name == "convert":
        tool_args["conversion_rate"] = conversion_rate

    result = tool_registry[tool_name].invoke(tool_args)

    # Keep conversation history as proper LangChain messages.
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"],
            name=tool_name,
        )
    )

    if tool_name == "get_conversion_factor":
        conversion_rate = result

In [130]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR? And based on that can you convert 10 usd to inr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 124, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E7OQqvZWeQ6B0m62KLz3ubcYrZIg0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fb3f9-c352-7562-97c6-b580d2aab3e9-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_3wSAm8EGMlZQ3k7rp1ObslhU', 'type': 'tool_call

In [131]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion factor between USD and INR is 95.7625. \n\nTherefore, 10 USD is equivalent to 957.625 INR.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 197, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-E7OQsFgsjNC3DGpSzDOfUjaZWbmLA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fb3f9-d04f-7aa1-8bfd-4e89cfe5cd20-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_tokens': 32, 'total_tokens': 229, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})